<a href="https://colab.research.google.com/github/anaptoro/mvp_wine_pairing/blob/main/Wine_pairing_colab_ready.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Colab setup: install dependencies
!pip -q install joblib pandas numpy matplotlib scikit-learn


In [2]:
import warnings
warnings.filterwarnings("ignore")


from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

from IPython.display import display


##### Data loading and pre processing

In [3]:
# Robust data loading for Colab or local runs
from pathlib import Path

DATA_FILENAME = "wine_food_pairings.csv"
RAW_DATA_URL = (
    "https://raw.githubusercontent.com/anaptoro/mvp_wine_pairing/main/data/"
    "wine_food_pairings.csv"
)

candidate_paths = [
    Path.cwd() / "data" / DATA_FILENAME,          # if notebook is run from repo root
    Path.cwd().parent / "data" / DATA_FILENAME,   # if notebook is run from notebooks/
    Path("/content/mvp_wine_pairing/data") / DATA_FILENAME,  # if repo was cloned in Colab
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is not None:
    print(f"Loading local dataset from: {DATA_PATH}")
    df = pd.read_csv(DATA_PATH)
else:
    print("Local dataset not found. Loading from GitHub raw URL...")
    df = pd.read_csv(RAW_DATA_URL)

display(df.head(5))


Local dataset not found. Loading from GitHub raw URL...


,wine_type,wine_category,food_item,food_category,cuisine,pairing_quality,quality_label,description
0,Syrah/Shiraz,Red,smoked sausage,Smoky BBQ,Spanish,2.0,Poor,Heuristic pairing assessment.
1,Grenache,Red,charcuterie board,Salty Snack,French,3.0,Neutral,Heuristic pairing assessment.
2,Madeira,Fortified,lemon tart,Dessert,French,4.0,Good,Acidic wine balances acidic food.
3,Cabernet Sauvignon,Red,roast lamb,Red Meat,Mexican,5.0,Excellent,Tannic red complements red meat fat.
4,Viognier,White,duck à l’orange,Poultry,Vietnamese,2.0,Poor,Heuristic pairing assessment.


In [4]:
### Defining the target column
RAW_TARGET_COL = "quality_label"

if RAW_TARGET_COL not in df.columns:
    raise ValueError(
        f"Column '{RAW_TARGET_COL}' not found. Available columns: {list(df.columns)}"
    )

print(df[RAW_TARGET_COL].value_counts(dropna=False).sort_index())

quality_label
Excellent    7163
Good         6451
Neutral      8108
Poor         6179
Terrible     7034
Name: count, dtype: int64


In [5]:
##grouping the target column in 3 classes, to easy the classification
def map_three_class_target(label):
    if pd.isna(label):
        return np.nan
    if label in ["Excellent", "Good"]:
        return "Good"
    if label == "Neutral":
        return "Okay"
    if label in ["Poor", "Terrible"]:
        return "Bad"
    return np.nan
df["target_class"] = df[RAW_TARGET_COL].apply(map_three_class_target)

#### Data cleaning
###### Dropping duplicated rows that can cause confusion, checking class distribution,and running a baseline model (Any final model should outperform the baseline, this shows our model is better than a random choice)

In [6]:

# =========================
# 1. DEFINE THE FEATURE COMBO
# =========================
combo_cols = ["wine_type", "wine_category","food_category", "cuisine",'food_item']

target_col = "target_class"

# =========================
# 2. COUNT LABELS PER COMBO
# =========================
summary = (
    df.groupby(combo_cols)[target_col]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

# ensure all expected label columns exist
for col in ["Bad", "Okay", "Good"]:
    if col not in summary.columns:
        summary[col] = 0

# =========================
# 3. MAJORITY LABEL
# =========================
summary["target_class"] = summary[["Bad", "Okay", "Good"]].idxmax(axis=1)
summary["total_count"] = summary[["Bad", "Okay", "Good"]].sum(axis=1)
summary["majority_count"] = summary[["Bad", "Okay", "Good"]].max(axis=1)
summary["majority_ratio"] = summary["majority_count"] / summary["total_count"]

# =========================
# 4. CLEANED DATASET
# =========================
df_clean_majority = summary[combo_cols + ["target_class", "total_count", "majority_count", "majority_ratio"]].copy()

print("Original rows:", len(df))
print("Cleaned rows (majority aggregated):", len(df_clean_majority))
print(df_clean_majority.head())

Original rows: 34935
Cleaned rows (majority aggregated): 8236
target_class wine_type wine_category food_category       cuisine  \
0             Albariño         White        Acidic  American BBQ   
1             Albariño         White        Acidic  American BBQ   
2             Albariño         White        Acidic  American BBQ   
3             Albariño         White        Acidic   Argentinian   
4             Albariño         White        Acidic   Argentinian   

target_class      food_item target_class  total_count  majority_count  \
0             caprese salad          Bad            5               2   
1              citrus salad         Good            4               2   
2                  gazpacho         Good            3               2   
3             caprese salad         Good            4               2   
4              citrus salad         Good            4               2   

target_class  majority_ratio  
0                   0.400000  
1                   0.500000

In [7]:
DROP_COLS = [RAW_TARGET_COL,'pairing_quality',"target_class","description"] #dropping those columns to remove bias

feature_cols = [c for c in df.columns if c not in DROP_COLS]

print("Feature columns:")
print(feature_cols)

Feature columns:
['wine_type', 'wine_category', 'food_item', 'food_category', 'cuisine']


In [8]:
#We dont have int columns here, but just letting it here in case we have
X = df[feature_cols].copy()
y = df["target_class"]

cat_cols = X.select_dtypes(include=["object","category","bool"]).columns.tolist()
num_cols = X.select_dtypes(include=["int"]).columns.tolist()


print("Cat cols:",cat_cols)
print("Num cols:",num_cols)

Cat cols: ['wine_type', 'wine_category', 'food_item', 'food_category', 'cuisine']
Num cols: []


In [9]:
## checking train test proportion
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train class distribution:")
print(y_train.value_counts(normalize=True))
print("Test class distribution:")
print(y_test.value_counts(normalize=True))


Train shape: (27948, 5)
Test shape: (6987, 5)
Train class distribution:
target_class
Good    0.389688
Bad     0.378202
Okay    0.232110
Name: proportion, dtype: float64
Test class distribution:
target_class
Good    0.389724
Bad     0.378274
Okay    0.232002
Name: proportion, dtype: float64


In [10]:
#### Baseline comparison
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_preds = baseline.predict(X_test)

baseline_acc = accuracy_score(y_test, baseline_preds)
baseline_f1 = f1_score(y_test, baseline_preds, average="macro")

print(f"Baseline accuracy: {baseline_acc:.4f}")
print(f"Baseline macro F1: {baseline_f1:.4f}")

Baseline accuracy: 0.3897
Baseline macro F1: 0.1870


In [ ]:


# =========================
# 1. FEATURES AND TARGET
# =========================
selected_features = [
    "wine_type",
    "wine_category",
    "food_category",
    "food_item",
]

X = df_clean_majority[selected_features].copy()
y = df_clean_majority["target_class"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))
print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))


# =========================
# 2. COLUMN TYPES
# =========================
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("\nCategorical columns:", cat_cols)
print("Numeric columns:", num_cols)


# =========================
# 3. PREPROCESSORS
# =========================
# For Tree and Naive Bayes: one-hot only
preprocessor_basic = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            num_cols,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat_cols,
        ),
    ],
    remainder="drop",
)

# For KNN and SVM: one-hot + scaling
preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            num_cols,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat_cols,
        ),
    ],
    remainder="drop",
)


# =========================
# 4. DEFINE MODELS + GRIDS
# =========================
models = {
    "knn": {
        "pipeline": Pipeline(
            steps=[
                ("preprocessor", preprocessor_scaled),
                ("scaler", StandardScaler(with_mean=False)),
                ("classifier", KNeighborsClassifier()),
            ]
        ),
        "param_grid": {
            "classifier__n_neighbors": [3, 5, 7, 11],
            "classifier__weights": ["uniform", "distance"],
            "classifier__metric": ["euclidean", "manhattan"],
        },
    },
    "decision_tree": {
        "pipeline": Pipeline(
            steps=[
                ("preprocessor", preprocessor_basic),
                ("classifier", DecisionTreeClassifier(random_state=42)),
            ]
        ),
        "param_grid": {
            "classifier__max_depth": [3, 5, 10, None],
            "classifier__min_samples_split": [2, 5, 10],
            "classifier__min_samples_leaf": [1, 2, 5],
            "classifier__class_weight": [None, "balanced"],
        },
    },
    "naive_bayes": {
        "pipeline": Pipeline(
            steps=[
                ("preprocessor", preprocessor_basic),
                ("classifier", MultinomialNB()),
            ]
        ),
        "param_grid": {
            "classifier__alpha": [0.1, 0.5, 1.0, 2.0],
        },
    },
    "svm": {
        "pipeline": Pipeline(
            steps=[
                ("preprocessor", preprocessor_scaled),
                ("scaler", StandardScaler(with_mean=False)),
                ("classifier", SVC(probability=True, random_state=42)),
            ]
        ),
        "param_grid": {
            "classifier__C": [0.5, 1, 5],
            "classifier__kernel": ["linear", "rbf"],
            "classifier__class_weight": [None, "balanced"],
        },
    },
}


# =========================
# 5. TRAIN, TUNE, EVALUATE
# =========================
results = []
best_estimators = {}

for model_name, model_info in models.items():
    print("\n" + "=" * 80)
    print(f"Training model: {model_name}")
    print("=" * 80)

    grid = GridSearchCV(
        estimator=model_info["pipeline"],
        param_grid=model_info["param_grid"],
        scoring="f1_macro",
        cv=5,
        n_jobs=-1,
        verbose=1,
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    best_estimators[model_name] = best_model

    y_pred = best_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")

    print(f"\nBest params for {model_name}:")
    print(grid.best_params_)

    print(f"\nBest CV macro F1: {grid.best_score_:.4f}")
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Test Macro F1: {macro_f1:.4f}")
    print(f"Test Weighted F1: {weighted_f1:.4f}")
    print("\nClassification report:")
    print(classification_report(y_test, y_pred))

    results.append(
        {
            "model": model_name,
            "best_params": grid.best_params_,
            "cv_macro_f1": grid.best_score_,
            "test_accuracy": acc,
            "test_macro_f1": macro_f1,
            "test_weighted_f1": weighted_f1,
        }
    )


# =========================
# 6. COMPARE RESULTS
# =========================
results_df = pd.DataFrame(results).sort_values(
    by=["test_macro_f1", "test_accuracy"],
    ascending=False,
).reset_index(drop=True)

print("\nFinal comparison:")
print(results_df[["model", "cv_macro_f1", "test_accuracy", "test_macro_f1", "test_weighted_f1"]])

best_model_name = results_df.iloc[0]["model"]
best_model = best_estimators[best_model_name]

print(f"\nBest overall model: {best_model_name}")

Train shape: (5765, 4)
Test shape: (2471, 4)

Train target distribution:
target_class
Bad     0.704076
Good    0.292108
Okay    0.003816
Name: proportion, dtype: float64

Test target distribution:
target_class
Bad     0.703764
Good    0.292189
Okay    0.004047
Name: proportion, dtype: float64

Categorical columns: ['wine_type', 'wine_category', 'food_category', 'food_item']
Numeric columns: []

Training model: knn
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Best params for knn:
{'classifier__metric': 'euclidean', 'classifier__n_neighbors': 5, 'classifier__weights': 'uniform'}

Best CV macro F1: 0.6308
Test Accuracy: 0.9235
Test Macro F1: 0.6041
Test Weighted F1: 0.9199

Classification report:
              precision    recall  f1-score   support

         Bad       0.92      0.98      0.95      1739
        Good       0.93      0.80      0.86       722
        Okay       0.00      0.00      0.00        10

    accuracy                           0.92      2471
   macro

In [ ]:
labels = ["Good", "Okay", "Bad"]

cm_norm = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")
disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=labels)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, values_format=".2f")
plt.title(f"Normalized Confusion Matrix - {model_name}")
plt.show()

#### Saving the best model


In [ ]:
# Save trained artifacts in a Colab-friendly output folder
OUTPUT_DIR = Path.cwd() / "artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, OUTPUT_DIR / "best_wine_pairing_model.pkl")
joblib.dump(selected_features, OUTPUT_DIR / "selected_features.pkl")

feature_options = {
    col: sorted(df_clean_majority[col].dropna().unique().tolist())
    for col in selected_features
}
joblib.dump(feature_options, OUTPUT_DIR / "feature_options.pkl")

print(f"Saved best model and metadata to: {OUTPUT_DIR.resolve()}")


In [ ]:
# Save dropdown mappings in the same output folder
wine_types_by_category = (
    df_clean_majority.groupby("wine_category")["wine_type"]
    .apply(lambda x: sorted(x.dropna().unique().tolist()))
    .to_dict()
)

wine_categories_by_type = (
    df_clean_majority.groupby("wine_type")["wine_category"]
    .apply(lambda x: sorted(x.dropna().unique().tolist()))
    .to_dict()
)

food_items_by_category = (
    df_clean_majority.groupby("food_category")["food_item"]
    .apply(lambda x: sorted(x.dropna().unique().tolist()))
    .to_dict()
)

joblib.dump(wine_categories_by_type, OUTPUT_DIR / "wine_categories_by_type.pkl")
joblib.dump(wine_types_by_category, OUTPUT_DIR / "wine_types_by_category.pkl")
joblib.dump(food_items_by_category, OUTPUT_DIR / "food_items_by_category.pkl")

print(f"Saved dropdown mappings to: {OUTPUT_DIR.resolve()}")
print("Files:")
for p in sorted(OUTPUT_DIR.glob("*.pkl")):
    print("-", p.name)


## Download exported artifacts

In [ ]:
# Optional: download all generated .pkl files from Colab
from google.colab import files

for p in sorted(OUTPUT_DIR.glob("*.pkl")):
    files.download(str(p))
